In [1]:
# NORTHSTAR — Solace Browser: Docs Pages QA (docs, docs/oauth3, docs/mcp, docs/quick-start)
# DNA: docs_qa = structure(HTML5) x oauth3(scopes+budgets+code) x mcp(protocol+tools) x quickstart(steps) x nav(breadcrumbs+toc)
# Template: solace-cli/notebooks/qa-templates/template-page-qa.ipynb
# Towers: T1(Masters) T6(Hackers) T16(AI Coders) T23(Web)
# Auth: 65537 | Papers: 46,47,49,50
#
# File-based probes — reads HTML/JS/CSS files directly (no running server needed)
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-docs-pages-qa"
NOTEBOOK_PATH = "notebooks/qa/27-docs-pages-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"

# Docs pages under test (with subdirectory paths)
DOCS_PAGES = {
    "docs": WEB / "docs.html",
    "docs/oauth3": WEB / "docs" / "oauth3.html",
    "docs/mcp": WEB / "docs" / "mcp.html",
    "docs/quick-start": WEB / "docs" / "quick-start.html",
}

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

def read_doc(name):
    return DOCS_PAGES[name].read_text(encoding="utf-8")

# Verify all pages exist
for name, path in DOCS_PAGES.items():
    assert path.exists(), f"{name} missing at {path}"

print(f"BOOTSTRAP: {len(DOCS_PAGES)} docs pages under test")
print(f"Pages: {list(DOCS_PAGES.keys())}")

BOOTSTRAP: 4 docs pages under test
Pages: ['docs', 'docs/oauth3', 'docs/mcp', 'docs/quick-start']


In [2]:
# P1: HTML Structure — all docs pages have proper HTML5 structure
print("=== P1: HTML Structure ===")

REQUIRED_STRUCTURE = {
    "doctype": r"<!DOCTYPE html>",
    "html_lang": r'<html\s+lang="[a-z]{2}"',
    "title": r"<title>.+</title>",
    "meta_charset": r'<meta\s+charset="[uU][tT][fF]-8"',
    "csp_header": r'Content-Security-Policy',
    "layout_js": r'layout\.js',
    "header_slot": r'id="header-slot"',
}

for page_name in DOCS_PAGES:
    html = read_doc(page_name)
    safe_name = page_name.replace("/", "_")
    for check_name, pattern in REQUIRED_STRUCTURE.items():
        found = bool(re.search(pattern, html, re.IGNORECASE))
        record(f"P1-{safe_name}-{check_name}", f"{page_name} has {check_name}", found,
               detail=f"Pattern {pattern!r} not found in {page_name}" if not found else "",
               tower_ref="T1,T16,T23")

print(f"\nP1 complete: {sum(1 for r in results if r['id'].startswith('P1') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P1'))} passed")

=== P1: HTML Structure ===
  PASS: docs has doctype
  PASS: docs has html_lang
  PASS: docs has title
  PASS: docs has meta_charset
  PASS: docs has csp_header
  PASS: docs has layout_js
  PASS: docs has header_slot
  PASS: docs/oauth3 has doctype
  PASS: docs/oauth3 has html_lang
  PASS: docs/oauth3 has title
  PASS: docs/oauth3 has meta_charset
  PASS: docs/oauth3 has csp_header
  PASS: docs/oauth3 has layout_js
  PASS: docs/oauth3 has header_slot
  PASS: docs/mcp has doctype
  PASS: docs/mcp has html_lang
  PASS: docs/mcp has title
  PASS: docs/mcp has meta_charset
  PASS: docs/mcp has csp_header
  PASS: docs/mcp has layout_js
  PASS: docs/mcp has header_slot
  PASS: docs/quick-start has doctype
  PASS: docs/quick-start has html_lang
  PASS: docs/quick-start has title
  PASS: docs/quick-start has meta_charset
  PASS: docs/quick-start has csp_header
  PASS: docs/quick-start has layout_js
  PASS: docs/quick-start has header_slot

P1 complete: 28/28 passed


In [3]:
# P2: docs/oauth3 — has code examples, scope tables, budget concepts
print("=== P2: OAuth3 Documentation Content ===")

oauth3_html = read_doc("docs/oauth3")

oauth3_checks = {
    "what_is_oauth3": (r'id="what"|What is OAuth3', "What is OAuth3 section"),
    "scoped_authority": (r'[Ss]coped authority', "Scoped authority concept"),
    "time_bounded": (r'[Tt]ime.bounded', "Time-bounded concept"),
    "budget_gated": (r'[Bb]udget.gated', "Budget-gated concept"),
    "approval_gate": (r'id="approval"|approval gate', "Approval gate section"),
    "budget_enforcement": (r'id="budgets"|Budget enforcement', "Budget enforcement section"),
    "code_examples": (r'<pre>.*\{.*\}.*</pre>', "Code examples (JSON budget configs)"),
    "five_budget_gates": (r'B1.*Policy|B2.*Limit|5 budget gates', "5 budget gates listed"),
    "evidence_trails": (r'id="evidence"|Evidence trails', "Evidence trails section"),
    "fail_closed": (r'id="fail-closed"|[Ff]ail.closed', "Fail-closed model section"),
    "revocation": (r'id="revoke"|[Rr]evocation', "Revocation section"),
    "faq": (r'id="faq"|FAQ', "FAQ section"),
    "toc": (r'class="docs-toc"', "Table of contents"),
    "breadcrumb": (r'class="docs-breadcrumb"', "Breadcrumb navigation"),
}

for name, (pattern, desc) in oauth3_checks.items():
    found = bool(re.search(pattern, oauth3_html, re.DOTALL))
    record(f"P2-oauth3-{name}", f"docs/oauth3 has {desc}", found,
           detail=f"Missing: {desc}" if not found else "", tower_ref="T6,T16,T23")

print(f"\nP2 complete: {sum(1 for r in results if r['id'].startswith('P2') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P2'))} passed")

=== P2: OAuth3 Documentation Content ===
  PASS: docs/oauth3 has What is OAuth3 section
  PASS: docs/oauth3 has Scoped authority concept
  PASS: docs/oauth3 has Time-bounded concept
  PASS: docs/oauth3 has Budget-gated concept
  PASS: docs/oauth3 has Approval gate section
  PASS: docs/oauth3 has Budget enforcement section
  PASS: docs/oauth3 has Code examples (JSON budget configs)
  PASS: docs/oauth3 has 5 budget gates listed
  PASS: docs/oauth3 has Evidence trails section
  PASS: docs/oauth3 has Fail-closed model section
  PASS: docs/oauth3 has Revocation section
  PASS: docs/oauth3 has FAQ section
  PASS: docs/oauth3 has Table of contents
  PASS: docs/oauth3 has Breadcrumb navigation

P2 complete: 14/14 passed


In [4]:
# P3: docs/mcp — has MCP protocol references, 7 tools, setup instructions
print("=== P3: MCP Documentation Content ===")

mcp_html = read_doc("docs/mcp")

mcp_checks = {
    "what_is_mcp": (r'id="what"|What is MCP', "What is MCP section"),
    "mcp_protocol": (r'Model Context Protocol|JSON-RPC|stdio', "MCP protocol reference"),
    "setup_section": (r'id="setup"|Setup', "Setup instructions"),
    "claude_code": (r'id="claude-code"|Claude Code', "Claude Code integration"),
    "cursor_codex": (r'id="cursor"|Cursor.*Codex', "Cursor/Codex integration"),
    "seven_tools": (r'id="tools"|7 browser tools', "7 browser tools section"),
    "tool_navigate": (r'<h3>navigate</h3>', "navigate tool"),
    "tool_click": (r'<h3>click</h3>', "click tool"),
    "tool_fill": (r'<h3>fill</h3>', "fill tool"),
    "tool_screenshot": (r'<h3>screenshot</h3>', "screenshot tool"),
    "tool_snapshot": (r'<h3>snapshot</h3>', "snapshot tool"),
    "tool_evaluate": (r'<h3>evaluate</h3>', "evaluate tool"),
    "tool_aria": (r'<h3>aria_snapshot</h3>', "aria_snapshot tool"),
    "code_examples": (r'<pre>.*mcpServers.*</pre>', "MCP config code examples"),
    "usage_examples": (r'id="examples"|Usage examples', "Usage examples section"),
    "oauth3_integration": (r'id="oauth3"|OAuth3.*budget', "OAuth3 integration section"),
    "toc": (r'class="docs-toc"', "Table of contents"),
    "breadcrumb": (r'class="docs-breadcrumb"', "Breadcrumb navigation"),
    "nav_links": (r'class="docs-nav-links"', "Navigation links to other docs"),
}

for name, (pattern, desc) in mcp_checks.items():
    found = bool(re.search(pattern, mcp_html, re.DOTALL))
    record(f"P3-mcp-{name}", f"docs/mcp has {desc}", found,
           detail=f"Missing: {desc}" if not found else "", tower_ref="T6,T16,T23")

print(f"\nP3 complete: {sum(1 for r in results if r['id'].startswith('P3') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P3'))} passed")

=== P3: MCP Documentation Content ===
  PASS: docs/mcp has What is MCP section
  PASS: docs/mcp has MCP protocol reference
  PASS: docs/mcp has Setup instructions
  PASS: docs/mcp has Claude Code integration
  PASS: docs/mcp has Cursor/Codex integration
  PASS: docs/mcp has 7 browser tools section
  PASS: docs/mcp has navigate tool
  PASS: docs/mcp has click tool
  PASS: docs/mcp has fill tool
  PASS: docs/mcp has screenshot tool
  PASS: docs/mcp has snapshot tool
  PASS: docs/mcp has evaluate tool
  PASS: docs/mcp has aria_snapshot tool
  PASS: docs/mcp has MCP config code examples
  PASS: docs/mcp has Usage examples section
  PASS: docs/mcp has OAuth3 integration section
  PASS: docs/mcp has Table of contents
  PASS: docs/mcp has Breadcrumb navigation
  PASS: docs/mcp has Navigation links to other docs

P3 complete: 19/19 passed


In [5]:
# P4: docs/quick-start — has step-by-step getting started content
print("=== P4: Quick Start Documentation Content ===")

qs_html = read_doc("docs/quick-start")

qs_checks = {
    "install_section": (r'id="install"|1\.\s*Install', "Install section"),
    "launch_section": (r'id="launch"|2\.\s*Launch', "Launch section"),
    "first_run": (r'id="first-run"|first automation', "First automation section"),
    "approval_flow": (r'id="approval"|approval flow', "Approval flow section"),
    "tutorial_section": (r'id="tutorial"|YinYang tutorial', "Tutorial section"),
    "next_steps": (r'id="next"|Next steps', "Next steps section"),
    "install_option_a": (r'Option A.*[Dd]ownload', "Option A: Download binary"),
    "install_option_b": (r'Option B.*source', "Option B: From source"),
    "code_examples": (r'<pre>.*git clone.*</pre>', "Code examples (git clone, pip install)"),
    "budget_mention": (r'[Bb]udget', "Budget enforcement mentioned"),
    "fail_closed_mention": (r'fail.closed|cancelled', "Fail-closed behavior mentioned"),
    "toc": (r'class="docs-toc"', "Table of contents"),
    "breadcrumb": (r'class="docs-breadcrumb"', "Breadcrumb navigation"),
    "nav_links": (r'class="docs-nav-links"', "Navigation links to other docs"),
    "link_to_mcp": (r'href="[^"]*docs/mcp"', "Link to MCP docs"),
    "link_to_oauth3": (r'href="[^"]*docs/oauth3"', "Link to OAuth3 docs"),
}

for name, (pattern, desc) in qs_checks.items():
    found = bool(re.search(pattern, qs_html, re.DOTALL))
    record(f"P4-qs-{name}", f"docs/quick-start has {desc}", found,
           detail=f"Missing: {desc}" if not found else "", tower_ref="T1,T16,T23")

# Also check docs hub (docs.html) links to all sub-pages
docs_hub = read_doc("docs")
hub_links = {
    "link_quickstart": (r'href="[^"]*docs/quick-start"', "Link to quick-start"),
    "link_mcp": (r'href="[^"]*docs/mcp"', "Link to MCP guide"),
    "link_oauth3": (r'href="[^"]*docs/oauth3"', "Link to OAuth3 safety"),
    "docs_cards": (r'class="docs-card', "Documentation cards grid"),
    "paper_network": (r'Paper Network|paper.*network', "Paper network section"),
}
for name, (pattern, desc) in hub_links.items():
    found = bool(re.search(pattern, docs_hub, re.DOTALL))
    record(f"P4-hub-{name}", f"docs hub has {desc}", found,
           detail=f"Missing: {desc}" if not found else "", tower_ref="T1,T23")

print(f"\nP4 complete: {sum(1 for r in results if r['id'].startswith('P4') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P4'))} passed")

=== P4: Quick Start Documentation Content ===
  PASS: docs/quick-start has Install section
  PASS: docs/quick-start has Launch section
  PASS: docs/quick-start has First automation section
  PASS: docs/quick-start has Approval flow section
  PASS: docs/quick-start has Tutorial section
  PASS: docs/quick-start has Next steps section
  PASS: docs/quick-start has Option A: Download binary
  PASS: docs/quick-start has Option B: From source
  PASS: docs/quick-start has Code examples (git clone, pip install)
  PASS: docs/quick-start has Budget enforcement mentioned
  PASS: docs/quick-start has Fail-closed behavior mentioned
  PASS: docs/quick-start has Table of contents
  PASS: docs/quick-start has Breadcrumb navigation
  PASS: docs/quick-start has Navigation links to other docs
  PASS: docs/quick-start has Link to MCP docs
  PASS: docs/quick-start has Link to OAuth3 docs
  PASS: docs hub has Link to quick-start
  PASS: docs hub has Link to MCP guide
  PASS: docs hub has Link to OAuth3 safet

In [6]:
# P5: Evidence Summary
print("=== P5: Evidence Summary ===\n")

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = total - passed
score = round((passed / total) * 100, 1) if total > 0 else 0.0

evidence_blob = json.dumps({
    "notebook": "27-docs-pages-qa",
    "timestamp": datetime.now().isoformat(),
    "total": total,
    "passed": passed,
    "failed": failed,
    "score": score,
    "results": results,
}, indent=2)

evidence_hash = hashlib.sha256(evidence_blob.encode()).hexdigest()

print(f"  Total probes : {total}")
print(f"  Passed       : {passed}")
print(f"  Failed       : {failed}")
print(f"  Score        : {score}%")
print(f"  Evidence hash: {evidence_hash[:16]}...")
print()

if failed > 0:
    print("FAILURES:")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['name']} -- {r['detail']}")

assert score >= MIN_SCORE, f"Score {score}% below minimum {MIN_SCORE}%"
print(f"\nVERDICT: PASS ({score}% >= {MIN_SCORE}% minimum)")

=== P5: Evidence Summary ===

  Total probes : 82
  Passed       : 82
  Failed       : 0
  Score        : 100.0%
  Evidence hash: 15ca35e7c94ce30c...


VERDICT: PASS (100.0% >= 70% minimum)
